# setup + imports - mostly baseline stuff

In [1]:
import os
import random

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

device = "cpu"
print(f"Using device: {device}")

/Users/ecaterina/practice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
from sklearn.model_selection import KFold
import time

In [3]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

In [4]:
# Load data from CSV files
train_data = pd.read_csv("./train_data.csv")
test_data = pd.read_csv("./test_data.csv")

# map columns 'c' and 'mask' to lists of integers
to_list = lambda x: list(map(int, x.split(",")))
train_data["c"] = train_data["c"].apply(to_list)
train_data["mask"] = train_data["mask"].apply(to_list)
test_data["c"] = test_data["c"].apply(to_list)

print(f"Loaded {len(train_data)} training examples and {len(test_data)} test examples.")

Loaded 5000 training examples and 500 test examples.


In [5]:
# Load model and tokenizer
model_path = "./pythia-14m"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).float().to(device)
model.eval()
print(f"Vocabulary size: {tokenizer.vocab_size}")

W1125 01:49:16.774000 78150 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|███████████████████████| 76/76 [00:00<00:00, 9767.05it/s]

Vocabulary size: 50254


In [6]:
# View the first example in human readable format.
example = train_data.iloc[0]
a = np.array(example['c'])[np.array(example['mask']) == 0]
b = np.array(example['c'])[np.array(example['mask']) == 1]

print(f"id: {example['id']}, a length: {len(a)}, b length: {len(b)}, c length: {len(example['c'])}")
print(f"Sentence A (first 30 tokens): {tokenizer.decode(a[:30])}...")
print(f"Sentence B (first 30 tokens): {tokenizer.decode(b[:30])}...")
print(f"Interleaved (first 30 tokens): {tokenizer.decode(example['c'][:30])}...")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPTNeoXTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


id: 0, a length: 131, b length: 180, c length: 311
Sentence A (first 30 tokens): the ancient spiritual system in oriental world stressed on simple and minimalist living so that there is less pressure on demand side and therefore, less propensity of...
Sentence B (first 30 tokens): however, we in the new millennium have to be more aware and better endowed to handle this all. we surely do not accept to stand in 'den...
Interleaved (first 30 tokens): however, we inthe the ancient new millennium spiritual have system in to ori be more aware andental world stressed better endowed on to simple handle this and...


# data exploration

In [7]:
lung = train_data['c'].str.len()
print(lung.value_counts())

c
124    55
147    53
138    52
114    52
131    50
       ..
409     1
86      1
397     1
565     1
474     1
Name: count, Length: 388, dtype: int64


In [8]:
nr1 = []
nr0 = []
for i in range(len(train_data)):
    row = train_data['mask'][i]
    n0 = 0
    n1 = 0
    for r in row:
        if r==1:
            n1 += 1
        else:
            n0 += 1
    nr1.append(n1)
    nr0.append(n0)

nr1 = np.array(nr1)
nr0 = np.array(nr0)
print((nr1+2)/(nr0+1))
print(nr1)
print(nr0)
print(nr1-nr0)

[1.37878788 0.49655172 1.16091954 ... 0.59036145 1.62857143 1.1       ]
[180  70  99 ...  47  55  53]
[131 144  86 ...  82  34  49]
[ 49 -74  13 ... -35  21   4]


# kinda feature enginnering?? + auxiliary stuff

In [ ]:
def nextlogprob(context):
    if len(context)==0:
        context = [tokenizer.eos_token_id]
    ids = torch.tensor([context[-64:]]).to(device)
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
    return torch.log_softmax(logits, dim=-1).cpu()

def predmask(c):
    A = [c[0]]
    B = []
    mask = [0]
    lpA = nextlogprob(A)         
    lpB = nextlogprob(B)          
    for tok in c[1:]:
        if lpA[tok] >= lpB[tok]:
            A.append(tok)
            mask.append(0)
            lpA = nextlogprob(A)  
        else:
            B.append(tok)
            mask.append(1)
            lpB = nextlogprob(B)
    return mask

In [ ]:
def buildprops(rows):
    props = []
    for i in rows:
        c = train_data['c'][i]
        m = train_data['mask'][i]
        a = [tokenizer.eos_token_id]
        b = [tokenizer.eos_token_id]
        for j in range(len(c)):
            if m[j] == 0:
                a.append(c[j])
            else:
                b.append(c[j])
        props.append(a[:256])
        props.append(b[:256])
    random.shuffle(props)
    return props

# da method

In [33]:
def beammask(c, K=5):
    # each beam: [total score, A, B, mask, predictions after A, predictions after B]
    beams = [[0.0, [c[0]], [], [0], nextlogprob([c[0]]), nextlogprob([])]]
    for tok in c[1:]:
        cands = []
        for score, A, B, mask, lpA, lpB in beams:
            cands.append((score + lpA[tok].item(), 0, A, B, mask, lpA, lpB))
            cands.append((score + lpB[tok].item(), 1, A, B, mask, lpA, lpB))
        cands.sort(key=lambda x: -x[0])
        beams = []
        for s, side, A, B, mask, lpA, lpB in cands[:K]:
            if side == 0:
                A2 = A + [tok]
                beams.append([s, A2, B, mask + [0], nextlogprob(A2), lpB])
            else:
                B2 = B + [tok]
                beams.append([s, A, B2, mask + [1], lpA, nextlogprob(B2)])
    best = beams[0]
    for b in beams:
        if b[0] > best[0]:
            best = b
    return best[3]

# finetuning

In [36]:
# luat din caietul de baseline

def evaluate(data, predict_fn):
    """Evaluate predict_fn on labeled data using symmetric mask accuracy."""
    total_acc = 0.0
    for example in data:
        true_mask = np.array(example['mask'])
        pred_mask = np.array(predict_fn(example))
        if len(pred_mask) != len(true_mask):
            continue
        acc = np.mean(pred_mask == true_mask)
        total_acc += max(acc, 1.0 - acc)
    return total_acc / len(data) if data else None

# Example: evaluate on training data
# accuracy = evaluate(train_data.to_dict('records'), predict_mask)
# print(f"Train accuracy: {accuracy:.4f}")

def predict_mask(example):
    return beammask(example['c'], 5)


In [24]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)
for fit_rows, val_rows in kf.split(train_data):
    break 
val = [train_data.iloc[i] for i in val_rows[:50]]

In [31]:
def oracle_acc(example):
    c = example['c']
    m = example['mask']
    A = []
    B = []
    lpA = nextlogprob(A)
    lpB = nextlogprob(B)
    correct = 0
    for j in range(len(c)):
        tok = c[j]
        if lpA[tok] >= lpB[tok]:
            guess = 0
        else:
            guess = 1
        if guess == m[j]:
            correct += 1
        # follow the TRUE answer, not the guess
        if m[j] == 0:
            A.append(tok)
            lpA = nextlogprob(A)
        else:
            B.append(tok)
            lpB = nextlogprob(B)
    return correct / len(c)

accs = []
for ex in val:
    accs.append(oracle_acc(ex))
print('with true contexts:', np.mean(accs))

with true contexts: 0.9005666945193905


In [37]:
start = time.time()
print('greedy:', evaluate(val[:10], lambda ex: predmask(ex['c'])), round(time.time() - start), 's')
start = time.time()
print('beam 5:', evaluate(val[:10], predict_mask), round(time.time() - start), 's')


greedy: 0.6198240655503235 9 s
beam 5: 0.6828659097038924 40 s


In [29]:
def finetune(rows):
    props = buildprops(rows)
    lr = 1e-4
    model.train()
    start = time.time()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for k in range(len(props)):
        ids = torch.tensor([props[k]]).to(device)
        loss = model(input_ids=ids, labels=ids).loss
        opt.zero_grad()
        loss.backward()
        opt.step()
        if k == 200:
            per_step = (time.time() - start) / 200
            print('  estimated training minutes:', round(per_step * len(props) / 60, 1))
        if k % 2000 == 0:
            print('  ', k, '/', len(props), 'loss', round(loss.item(), 3))

    model.eval()

def scorerows(rows):
    total = 0.0
    for i in rows:
        true_mask = np.array(train_data['mask'][i])
        pred_mask = np.array(predict_mask(train_data.iloc[i]))
        acc = np.mean(pred_mask == true_mask)
        total += max(acc, 1.0 - acc)
    return total / len(rows)

In [25]:
print('before:', evaluate(val, predict_mask))

before: 0.5796837415159899


In [28]:
finetune(fit_rows)
print('after:', evaluate(val, predict_mask))

   0 / 10000 loss 5.016
  estimated training minutes: 9.5
   2000 / 10000 loss 4.437
   4000 / 10000 loss 4.187
   6000 / 10000 loss 4.896
   8000 / 10000 loss 4.887
after: 0.6086965733191737


# metoda kinda

In [ ]:
'''
pred = []
for i in range(len(test_data)):
    prop = test_data['c'][i]
    pred.append(predmask(prop))

print(pred)
'''

In [ ]:
import time
total_tokens = sum(len(c) for c in test_data['c'])

start = time.time()
m = predmask(test_data['c'][0])
per_token = (time.time() - start) / len(test_data['c'][0])
print('estimated minutes for the test set:', round(per_token * total_tokens / 60, 1))

# evaluating

In [ ]:
small = train_data.head(50).to_dict('records')
print(evaluate(small, predict_mask))

In [ ]:
sw = []
for m in train_data['mask']:
    for i in range(1, len(m)):
        sw.append(m[i] != m[i - 1])
p_switch = np.mean(sw)
log_stay = np.log(1 - p_switch)
log_switch = np.log(p_switch)
print(p_switch)

# submission

In [41]:
import json
done = {}
if os.path.exists('preds.json'):
    with open('preds.json') as f:
        done = json.load(f)

start = time.time()
for i in range(len(test_data)):
    key = str(test_data['id'][i])
    if key in done:
        continue
    done[key] = beammask(test_data['c'][i], 5)
    if len(done) % 25 == 0:
        with open('preds.json', 'w') as f:
            json.dump(done, f)
        print(len(done), '/', len(test_data), round((time.time() - start) / 60, 1), 'min')
with open('preds.json', 'w') as f:
    json.dump(done, f)

375 / 500 1.8 min
400 / 500 3.4 min
425 / 500 5.0 min
450 / 500 6.6 min
475 / 500 8.4 min
500 / 500 10.0 min


In [42]:
with open('submission.csv', 'w') as f:
    f.write('subtaskID,datapointID,answer\n')
    for i in range(len(test_data)):
        key = str(test_data['id'][i])
        answer = '"' + ','.join(str(m) for m in done[key]) + '"'
        f.write(f'"1","{key}",{answer}\n')

sub = pd.read_csv('submission.csv')
print(sub.shape)                                                   # (500, 3)
print(len(sub['answer'][0].split(',')), len(test_data['c'][0]))    # must match

(500, 3)
359 359
